# Loss Functions

**Topic:** Measuring Model Error

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** what a loss function measures and why every model needs one
- **Explain** how MSE and MAE treat outliers differently and why that matters for real-world data
- **Interpret** how changing the loss function changes what the model optimizes for

> **Tip:** First move the **Outlier offset** slider to +15 and watch how the regression line shifts with MSE vs MAE. MSE chases the outlier; MAE mostly ignores it. That difference determines which loss to use on messy real-world data.

---
## How we got here

In **06_cross_validation.ipynb** we measured model performance after training. But the model itself uses a loss function during training to measure its own performance and update its parameters.

This connects directly to:
- **math/calculus/08_the_cost_function.ipynb** — you derived the cost function mathematically and understood what gradient descent minimizes

Here we make that abstract function concrete: what does minimizing MSE vs MAE actually do to the fitted line?

---
## Why this matters for data science

The loss function defines what "better" means for your model. If you use MSE and your data has outliers, your model will bend itself toward those outliers because squaring the error makes them dominate the loss.

Choosing the wrong loss function is one of the most common and least visible mistakes in ML. The model trains without errors, looks reasonable, and then fails in production because it was optimizing the wrong thing all along.

---
## Try it yourself

In [2]:
np.random.seed(42)
from sklearn.linear_model import HuberRegressor, QuantileRegressor

X_raw = np.linspace(0, 10, 40)
y_raw = 2*X_raw + 1 + np.random.normal(0, 2, 40)

out = Output()

loss_dd = Dropdown(
    options=["MSE", "MAE", "Huber"],
    value="MSE",
    description="Loss function:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="380px"))

outlier_slider = FloatSlider(value=0.0, min=-15.0, max=15.0, step=0.5,
    description="Outlier offset:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="380px"))

DELTA = 1.35  # Huber threshold

def fit_line(loss_name, Xr, y):
    """Fit the line by ACTUALLY minimizing the chosen loss (not always OLS)."""
    if loss_name == "MSE":
        m = LinearRegression().fit(Xr, y)
    elif loss_name == "MAE":
        m = QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs").fit(Xr, y)
    else:  # Huber
        m = HuberRegressor(epsilon=DELTA, alpha=0.0).fit(Xr, y)
    return m.predict(Xr), float(m.coef_[0]), float(m.intercept_)

def loss_of(name, r):
    """Per-point loss contribution as a function of residual r."""
    if name == "MSE":
        return r**2
    if name == "MAE":
        return np.abs(r)
    a = np.abs(r)
    return np.where(a <= DELTA, 0.5*r**2, DELTA*(a - 0.5*DELTA))

def render(loss_name, outlier_offset):
    Xr = X_raw.reshape(-1, 1)
    y_mod = y_raw.copy()
    y_mod[-1] = y_mod[-1] + outlier_offset       # last point is the outlier
    out_i = len(X_raw) - 1

    # Selected-loss fit (this is what now actually moves) + OLS reference
    y_sel, coef, intercept = fit_line(loss_name, Xr, y_mod)
    y_mse, _, _            = fit_line("MSE", Xr, y_mod)

    resid = y_mod - y_sel
    loss_vals = loss_of(loss_name, resid)
    total_loss = float(loss_vals.sum())

    # ---------- LEFT PANEL: data + fitted line ----------
    normal = np.arange(len(X_raw) - 1)
    left_traces = [
        go.Scatter(x=X_raw[normal], y=y_mod[normal], mode="markers",
            marker=dict(color=PALETTE["muted"], size=8, opacity=0.7), name="Data"),
        go.Scatter(x=[X_raw[out_i]], y=[y_mod[out_i]], mode="markers",
            marker=dict(color=PALETTE["secondary"], size=14, line=dict(width=1, color="white")),
            name="Outlier"),
        go.Scatter(x=X_raw, y=y_mse, mode="lines",
            line=dict(color=PALETTE["muted"], width=1.5, dash="dash"),
            name="MSE reference"),
        go.Scatter(x=X_raw, y=y_sel, mode="lines",
            line=dict(color=PALETTE["primary"], width=2.8),
            name=f"{loss_name} fit"),
        # drop-line: how far the outlier sits from the selected line
        go.Scatter(x=[X_raw[out_i], X_raw[out_i]], y=[y_mod[out_i], y_sel[out_i]],
            mode="lines", line=dict(color=PALETTE["secondary"], width=1.5, dash="dot"),
            showlegend=False),
    ]
    lay_left = base_layout(
        title=f"{loss_name} fit — slope={coef:.2f}  |  offset={outlier_offset:+.1f}",
        xaxis_title="Feature", yaxis_title="Target")
    lay_left.update(width=470, height=400)
    fig_left = go.Figure(data=left_traces, layout=lay_left)

    # ---------- RIGHT PANEL: the loss function itself ----------
    R = max(float(np.max(np.abs(resid))), 1.0) * 1.1
    r_grid = np.linspace(-R, R, 240)
    curve = loss_of(loss_name, r_grid)
    right_traces = [
        go.Scatter(x=r_grid, y=curve, mode="lines",
            line=dict(color=PALETTE["primary"], width=2.5), name=f"{loss_name}(r)"),
        go.Scatter(x=resid[normal], y=loss_vals[normal], mode="markers",
            marker=dict(color=PALETTE["muted"], size=7, opacity=0.7), name="Points"),
        go.Scatter(x=[resid[out_i]], y=[loss_vals[out_i]], mode="markers",
            marker=dict(color=PALETTE["secondary"], size=14, line=dict(width=1, color="white")),
            name=f"Outlier (loss={loss_vals[out_i]:.1f})"),
    ]
    lay_right = base_layout(
        title=f"{loss_name} loss shape — total = {total_loss:.1f}",
        xaxis_title="Residual  (y − ŷ)", yaxis_title="Loss contribution")
    lay_right.update(width=470, height=400)
    fig_right = go.Figure(data=right_traces, layout=lay_right)

    with out:
        clear_output(wait=True)
        display(HBox([go.FigureWidget(fig_left), go.FigureWidget(fig_right)]))

def on_change(change): render(loss_dd.value, outlier_slider.value)
loss_dd.observe(on_change, names="value")
outlier_slider.observe(on_change, names="value")
display(VBox([loss_dd, outlier_slider, out]))
render("MSE", 0.0)

---
## What's happening?

Think of a loss function as a scoring rule: it measures how far each prediction lands from the truth, and training is the search for the line that makes the *total* score as small as possible. The crucial idea this demo makes visible is that **different scoring rules pick different lines** — so the loss function you choose silently decides what your model considers a good fit.

The widget shows this from two angles at once:

**Left panel — where the line lands.** The bold line is fit by *actually minimizing the selected loss*, and the dashed gray line is the MSE fit kept as a fixed reference. Drag the **Outlier offset** to +15 and watch the gap between them:

- **MSE** chases the outlier. Because squaring makes a distant point's error enormous, the line tilts upward to reduce that one giant penalty — dragging the fit away from the other 39 points. The slope readout climbs.
- **MAE** mostly ignores it. Absolute error grows only linearly, so one far point can't out-shout the crowd. The line stays on the honest trend, and the gap from the dashed MSE reference *is* the robustness story.
- **Huber** lands in between — it follows MSE for the well-behaved points and switches to MAE-like restraint for the outlier.

**Right panel — why.** This is the loss function's own shape: residual on the x-axis, the penalty that residual earns on the y-axis. Every data point is plotted on the curve at its own residual, with the outlier marked. This panel explains the left one:

- **MSE is a parabola.** The outlier sits far out on the x-axis, and the curve has lifted it sky-high — its single contribution dwarfs everything else, which is *why* the line on the left bent toward it.
- **MAE is a V.** The same far-out residual lands on the gentle straight arm, contributing only modestly. No single point can dominate, so the line stays put.
- **Huber is a parabola with its steep ends sawn off** at the threshold δ — quadratic near zero, linear past δ. The elbow is where it stops punishing large errors quadratically.

The throughline: **the shape of the curve on the right is the cause; the position of the line on the left is the effect.**

| Loss function | Formula | Shape | Sensitive to outliers? | Used for | sklearn estimator |
|---|---|---|---|---|---|
| MSE | mean((y − ŷ)²) | Parabola | Yes — heavily | Regression with clean data | `LinearRegression` |
| MAE | mean(\|y − ŷ\|) | V | No — robust | Regression with outliers | `QuantileRegressor(quantile=0.5)` |
| Huber | hybrid of MSE+MAE | Parabola with linear tails | Partially | Robust regression | `HuberRegressor` |
| Log loss | −mean(y·log(ŷ)) | Convex curve | N/A | Binary classification | `LogisticRegression` |
| Hinge | max(0, 1 − y·ŷ) | Hinge | N/A | SVM classification | `LinearSVC` |

> **One caution:** don't compare the *total-loss numbers across* losses — MSE is in squared units and MAE in absolute units, so a smaller MAE total than MSE total means nothing. Compare the *line's position and slope* across losses; compare *totals* only within one loss as you move the slider.

---
## Real-world example: Predicting delivery times

An e-commerce platform predicts delivery times. A few customers had extreme delays (warehouse fire, holiday surge). With MSE, the model chases those extreme cases and systematically over-predicts delivery time for the 95% of normal orders.

Switching to MAE produces a model that is accurate for typical deliveries and less influenced by rare extreme events — a better fit for the actual business need.

---
## Key takeaway

> **The loss function defines what the model optimizes for — choosing the wrong one means training a model that is technically correct but wrong for your problem.**

---
*Next up: Gradient Descent — the engine that actually minimizes the loss function*